# Autonomous Incident Response System for AWS
---

In this example, we will build an Agentic system to respond to incidents in your AWS accounts. This is a multi-agent system that composes 4 main components: 

1. **Monitoring**: This composes of a couple of aspects which includes monitoring CloudWatch alarms on the pre-built alarms that have already been set in your account. This might include high `CPU` usage, unhealthy load balancers, SageMaker instance cost allocations, etc. This would also include observing logs from different services from your account and classifying those logs into `Critical` (for example service down, `CPU`>`90%`), `Warning` (for example, latency > threshold, or if something goes beyond a threshold for a specific service) and `Informational` (for example, routine backups, information on various running applications in the AWS account, etc.).

1. **Diagnosis**: This includes diagnosis events that are seen through the monitoring agent. This can include querying `AWS` CloudTrail for additional data, X-Ray data and document these findings in reports that can be saved and used later in the resolution process. This would contain information only on the errors and the different services that need a resolution.

1. **Resolution**: This portion of the solution will be triggered by a diagnosis done from the step before. Once the diagnoses is done with the clear report, then this portion starts to remediate certain actions, such as adjusting EC2 auto-scaling group capacities, invoking functions to rollback deployments, etc. This agent is an essential part of the system since it will be using AWS `API`s in real time to manage the resources.

1. **Communication**: Last, this agent is responsible for keeping track of updates, creating and updating tickets in Jira, sending real time notifications to Slack with the incident details and resolution updates.

This solution will also contain aspects for observabilitiy and tracing but without further ado, let's get right into it.

In [1]:
# LangGraph is a low level orchestration framework for building controllable agents. 
# While langchain provides integrations and composable components to streamline LLM application development, 
# the LangGraph library enables agent orchestration, long term memory, human in the loop and customizable architectures.

In [12]:
import boto3
import logging
from datetime import datetime, timedelta
from typing import Annotated, List, Dict, Any
from typing_extensions import TypedDict
# import langgraph relevant libraries
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

# Import the memory saver to save in checkpoint and in some thread to retain agent's memory
from langgraph.checkpoint.memory import MemorySaver

# langchain imports
from langchain_aws.chat_models import ChatBedrockConverse
from langchain_core.tools import tool

In [13]:
# Create a logger
logger = logging.getLogger()
logger.setLevel(logging.INFO)

# Remove existing handlers
logger.handlers.clear()

# Add a simple handler
handler = logging.StreamHandler()
formatter = logging.Formatter('[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s')
handler.setFormatter(formatter)
logger.addHandler(handler)

In [14]:
# define the constants
AMAZON_NOVA_PRO_MODEL_ID: str = 'amazon.nova-pro-v1:0'

In [15]:
# create the state schema which will track the AWS incident responses
import json
import time
from typing import List

class AWSState(TypedDict):
    """
    This class contains the state for the AWS incident response.
    """
    # the messages adds it to the list of messages that preserves
    # the chat history
    messages: Annotated[list, add_messages]
    # This incident contains the incident details
    incident: dict
    # Contains the results from the diagnosis
    diagnosis: dict
    # Contains the list of resolution actions that needs to be performed
    actions: list
    # to store communication records and the status of the incident
    notifications: list
    status: str

In [16]:
# Let's now initialize our LLM, we will use Amazon Nova for this example
llm = ChatBedrockConverse(
    model_id=AMAZON_NOVA_PRO_MODEL_ID,
    temperature = 0.1,
)

[2025-03-14 15:53:27,367] p66282 {credentials.py:1352} INFO - Found credentials in shared credentials file: ~/.aws/credentials


### Set up monitoring tools
---

Next, we will set up some monitoring tools that the multi-agent system will use to monitor the CloudWatch logs, alarms, and status of EC2 instances. We will bind these tools to our LLM and use that in the multi-agent orchestration.

In [ ]:
# ---------------- MONITORING TOOLS ---------------------
def check_cloudwatch_alarms(
    filter_state: str = "ALARM",
    region: str = None,
    alarm_name_prefix: str = None,
    max_records: int = 100
) -> List[Dict]:
    """
    Check AWS CloudWatch alarms and return alarms in the specified state.
    
    Args:
        filter_state: Filter alarms by state ('ALARM', 'INSUFFICIENT_DATA', 'OK')
        region: AWS region to check (defaults to configured region)
        alarm_name_prefix: Filter alarms by name prefix
        max_records: Maximum number of records to return
        
    Returns:
        List of alarm details as dictionaries
    """
    # Create a boto3 CloudWatch client
    session = boto3.Session(region_name=region) if region else boto3.Session()
    cloudwatch = session.client('cloudwatch')
    
    # Parameters for the API call
    params = {
        'StateValue': filter_state,
        'MaxRecords': max_records
    }
    
    # Add optional name prefix if provided
    if alarm_name_prefix:
        params['AlarmNamePrefix'] = alarm_name_prefix
    
    # Call AWS API to get alarms, this will give us the status of the alarms
    response = cloudwatch.describe_alarms(**params)
    
    # Format the response data
    alarms = []
    # Now, let's filter out the response data based on the current cloudwatch alarms
    for alarm in response.get('MetricAlarms', []):
        # Extract the resource ID from dimensions if available
        resource_id = None
        for dimension in alarm.get('Dimensions', []):
            if dimension['Name'] in ['InstanceId', 'DBInstanceIdentifier', 'LoadBalancerName']:
                resource_id = dimension['Value']
                break
        
        # Format each alarm's data
        alarm_data = {
            "AlarmName": alarm['AlarmName'],
            "AlarmDescription": alarm.get('AlarmDescription', ''),
            "StateValue": alarm['StateValue'],
            "StateReason": alarm.get('StateReason', ''),
            "Timestamp": alarm.get('StateUpdatedTimestamp', datetime.now()).isoformat(),
            "Region": session.region_name,
            "Threshold": alarm.get('Threshold'),
            "CurrentValue": None,  # This requires an additional API call (see below)
            "ResourceId": resource_id
        }
        
        alarms.append(alarm_data)
    
    # If you want to include the current metric values, you'll need additional API calls
    # This is optional but makes the data more useful
    for alarm in alarms:
        try:
            if 'MetricName' in alarm and 'Namespace' in alarm:
                # Get the most recent datapoint for this metric
                metric_data = cloudwatch.get_metric_statistics(
                    Namespace=alarm['Namespace'],
                    MetricName=alarm['MetricName'],
                    Dimensions=alarm.get('Dimensions', []),
                    StartTime=datetime.now() - timedelta(hours=1),
                    EndTime=datetime.now(),
                    Period=60,
                    Statistics=['Average']
                )
                
                # Extract the most recent value if available
                datapoints = metric_data.get('Datapoints', [])
                if datapoints:
                    latest = max(datapoints, key=lambda x: x['Timestamp'])
                    alarm['CurrentValue'] = latest.get('Average')
        except Exception as e:
            pass
    return alarms

def get_ec2_metrics(
    instance_id: str,
    metric_name: str = "CPUUtilization",
    statistic: str = "Average",
    period: int = 300,
    duration_hours: int = 1,
    region: str = None
) -> Dict:
    """
    Get CloudWatch metrics for a specific EC2 instance.
    
    Args:
        instance_id: The EC2 instance ID (e.g., i-1234567890abcdef0)
        metric_name: Name of the EC2 metric to retrieve:
                     CPUUtilization, NetworkIn, NetworkOut, DiskReadBytes,
                     DiskWriteBytes, StatusCheckFailed, etc.
        statistic: Statistic to retrieve (Average, Maximum, Minimum, Sum, SampleCount)
        period: Time period between data points in seconds (default: 300)
        duration_hours: How many hours of data to retrieve (default: 1)
        region: AWS region (defaults to configured region)
        
    Returns:
        Dictionary containing EC2 metric data or error information
    """
    # Create a boto3 CloudWatch client
    session = boto3.Session(region_name=region) if region else boto3.Session()
    cloudwatch = session.client('cloudwatch')
    ec2 = session.client('ec2')
    
    # Validate that this is a valid EC2 instance
    try:
        instance_info = ec2.describe_instances(InstanceIds=[instance_id])
        if not instance_info['Reservations'] or not instance_info['Reservations'][0]['Instances']:
            return {
                "error": f"EC2 instance {instance_id} not found",
                "statusCode": 404
            }
    except Exception as e:
        return {
            "error": f"Error validating EC2 instance: {str(e)}",
            "statusCode": 400
        }
    
    # Calculate start and end times
    end_time = datetime.now()
    start_time = end_time - timedelta(hours=duration_hours)
    
    try:
        # Get metric data using get_metric_data for better performance
        response = cloudwatch.get_metric_data(
            MetricDataQueries=[
                {
                    'Id': 'ec2metric',
                    'MetricStat': {
                        'Metric': {
                            'Namespace': 'AWS/EC2',
                            'MetricName': metric_name,
                            'Dimensions': [
                                {
                                    'Name': 'InstanceId',
                                    'Value': instance_id
                                }
                            ]
                        },
                        'Period': period,
                        'Stat': statistic
                    }
                }
            ],
            StartTime=start_time,
            EndTime=end_time
        )
        
        # Extract and format results
        results = response['MetricDataResults'][0]
        
        # Get instance details for additional context
        instance_details = instance_info['Reservations'][0]['Instances'][0]
        instance_type = instance_details.get('InstanceType', 'Unknown')
        instance_state = instance_details.get('State', {}).get('Name', 'Unknown')
        
        # If we got data, format it nicely
        if results['Values']:
            # Ensure timestamps and values are paired and sorted chronologically
            data_points = list(zip(results['Timestamps'], results['Values']))
            data_points.sort(key=lambda x: x[0])  # Sort by timestamp
            
            timestamps = [t.timestamp() for t, _ in data_points]
            values = [v for _, v in data_points]
            
            # Calculate some basic statistics
            avg_value = sum(values) / len(values) if values else 0
            max_value = max(values) if values else 0
            min_value = min(values) if values else 0
            
            return {
                "ResourceId": instance_id,
                "InstanceType": instance_type,
                "InstanceState": instance_state,
                "MetricName": metric_name,
                "Statistic": statistic,
                "Timestamps": timestamps,
                "Values": values,
                "Unit": results.get('Unit', 'None'),
                "Region": session.region_name,
                "Summary": {
                    "Average": avg_value,
                    "Maximum": max_value,
                    "Minimum": min_value,
                    "DataPoints": len(values)
                }
            }
        else:
            # No data points, but return context about the instance
            return {
                "ResourceId": instance_id,
                "InstanceType": instance_type,
                "InstanceState": instance_state,
                "MetricName": metric_name,
                "Status": "No data points available in the specified time range",
                "Region": session.region_name
            }
            
    except Exception as e:
        return {
            "error": f"Error fetching CloudWatch metrics: {str(e)}",
            "statusCode": 500
        }